In [19]:
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

from scipy.optimize import minimize


# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

df = pd.read_csv("/content/adv.csv")
df.dropna(axis=0,inplace=True)
print("Original Dataset Shape:")
print(df.shape)

print("\nFirst 5 Rows:")
print(df.head())


# ------------------------------------------------------------
# 3. REMOVE EMPTY ROWS
# ------------------------------------------------------------

data = df.dropna(
    subset=["TV", "Radio", "Newspaper", "Sales"]
).copy()

print("\nDataset Shape After Removing Empty Rows:")
print(data.shape)


# ------------------------------------------------------------
# 4. DEFINE FEATURES AND TARGET
# ------------------------------------------------------------

X = data[
    ["TV", "Radio", "Newspaper"]
]

y = data["Sales"]


# ------------------------------------------------------------
# 5. TRAIN / TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# ------------------------------------------------------------
# 6. NORMALIZATION
# ------------------------------------------------------------

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)


print("\nNormalized Training Data:")
print(X_train_scaled[:5])


# ============================================================
# PART A
# SKLEARN LINEAR REGRESSION
# ============================================================

print("\n======================================")
print("SKLEARN LINEAR REGRESSION")
print("======================================")


# ------------------------------------------------------------
# 7. CREATE LINEAR REGRESSION MODEL
# ------------------------------------------------------------

model = LinearRegression()


# ------------------------------------------------------------
# 8. TRAIN MODEL
# ------------------------------------------------------------

model.fit(
    X_train_scaled,
    y_train
)


# ------------------------------------------------------------
# 9. PREDICTION
# ------------------------------------------------------------

y_train_pred = model.predict(
    X_train_scaled
)

y_test_pred = model.predict(
    X_test_scaled
)


# ------------------------------------------------------------
# 10. DISPLAY PARAMETERS
# ------------------------------------------------------------

print("\nIntercept:")
print(model.intercept_)

print("\nCoefficients:")

for feature, coefficient in zip(
    X.columns,
    model.coef_
):

    print(
        feature,
        "=",
        coefficient
    )


# ------------------------------------------------------------
# 11. MODEL PERFORMANCE
# ------------------------------------------------------------

train_mse = mean_squared_error(
    y_train,
    y_train_pred
)

test_mse = mean_squared_error(
    y_test,
    y_test_pred
)

train_r2 = r2_score(
    y_train,
    y_train_pred
)

test_r2 = r2_score(
    y_test,
    y_test_pred
)


print("\nTraining MSE:")
print(train_mse)

print("\nTesting MSE:")
print(test_mse)

print("\nTraining R²:")
print(train_r2)

print("\nTesting R²:")
print(test_r2)


# ============================================================
# PART B
# COST FUNCTION
# ============================================================

print("\n======================================")
print("COST CALCULATION")
print("======================================")


# Cost function ONLY
# No manual gradient
# No manual parameter update

def cost_function(theta):

    intercept = theta[0]

    coefficients = theta[1:]

    prediction = (
        intercept
        + X_train_scaled @ coefficients
    )

    error = (
        y_train.to_numpy()
        - prediction
    )

    cost = (
        np.mean(error ** 2)
        / 2
    )

    return cost


# ------------------------------------------------------------
# INITIAL PARAMETERS
# ------------------------------------------------------------

initial_theta = np.zeros(
    X_train_scaled.shape[1] + 1
)

print("\nInitial Parameters:")
print(initial_theta)


# ------------------------------------------------------------
# INITIAL COST
# ------------------------------------------------------------

initial_cost = cost_function(
    initial_theta
)

print("\nInitial Cost:")
print(initial_cost)


# ============================================================
# PART C
# COST MINIMIZATION USING SCIPY
# ============================================================

print("\n======================================")
print("SCIPY COST MINIMIZATION")
print("======================================")


# ------------------------------------------------------------
# scipy.optimize.minimize()
#
# IMPORTANT:
# No manual gradient
# No manual gradient descent
# No manual parameter update
#
# SciPy performs the optimization.
# ------------------------------------------------------------

result = minimize(

    cost_function,

    initial_theta,

    method="BFGS"

)


# ------------------------------------------------------------
# 12. OPTIMIZED PARAMETERS
# ------------------------------------------------------------

optimized_theta = result.x

print("\nOptimized Parameters:")

print(optimized_theta)


# ------------------------------------------------------------
# 13. MINIMUM COST
# ------------------------------------------------------------

minimum_cost = result.fun

print("\nMinimum Cost:")
print(minimum_cost)


# ------------------------------------------------------------
# 14. OPTIMIZATION STATUS
# ------------------------------------------------------------

print("\nOptimization Successful:")
print(result.success)

print("\nOptimization Message:")
print(result.message)

print("\nNumber of Iterations:")
print(result.nit)


# ============================================================
# PART D
# USE OPTIMIZED PARAMETERS FOR PREDICTION
# ============================================================

print("\n======================================")
print("OPTIMIZED MODEL")
print("======================================")


optimized_intercept = optimized_theta[0]

optimized_coefficients = optimized_theta[1:]


# ------------------------------------------------------------
# Prediction using optimized parameters
# ------------------------------------------------------------

optimized_train_prediction = (
    optimized_intercept
    + X_train_scaled @ optimized_coefficients
)

optimized_test_prediction = (
    optimized_intercept
    + X_test_scaled @ optimized_coefficients
)


# ------------------------------------------------------------
# 15. FINAL MSE
# ------------------------------------------------------------

optimized_train_mse = mean_squared_error(
    y_train,
    optimized_train_prediction
)

optimized_test_mse = mean_squared_error(
    y_test,
    optimized_test_prediction
)


# ------------------------------------------------------------
# 16. FINAL R²
# ------------------------------------------------------------

optimized_train_r2 = r2_score(
    y_train,
    optimized_train_prediction
)

optimized_test_r2 = r2_score(
    y_test,
    optimized_test_prediction
)


# ============================================================
# FINAL RESULTS
# ============================================================

print("\n======================================")
print("FINAL RESULTS")
print("======================================")


print("\nOptimized Intercept:")
print(optimized_intercept)


print("\nOptimized Coefficients:")

for feature, coefficient in zip(
    X.columns,
    optimized_coefficients
):

    print(
        feature,
        "=",
        coefficient
    )


print("\nMinimum Cost:")
print(minimum_cost)


print("\nTraining MSE:")
print(optimized_train_mse)


print("\nTesting MSE:")
print(optimized_test_mse)


print("\nTraining R²:")
print(optimized_train_r2)


print("\nTesting R²:")
print(optimized_test_r2)


# ============================================================
# COMPARISON
# ============================================================

print("\n======================================")
print("SKLEARN vs SCIPY")
print("======================================")


print("\nSklearn Intercept:")
print(model.intercept_)

print("\nSciPy Optimized Intercept:")
print(optimized_intercept)


print("\nSklearn Coefficients:")
print(model.coef_)

print("\nSciPy Optimized Coefficients:")
print(optimized_coefficients)


print("\nSklearn Training Cost:")
print(train_mse / 2)

print("\nSciPy Minimum Cost:")
print(minimum_cost)

Original Dataset Shape:
(200, 4)

First 5 Rows:
      TV  Radio  Newspaper  Sales
1  230.1   37.8       69.2   22.1
3   44.5   39.3       45.1   10.4
5   17.2   45.9       69.3    9.3
7  151.5   41.3       58.5   18.5
9  180.8   10.8       58.4   12.9

Dataset Shape After Removing Empty Rows:
(200, 4)

Normalized Training Data:
[[-0.40424839 -1.02823707 -0.33767538]
 [ 0.32060772 -0.91982774 -1.16143931]
 [-1.27051084  0.2591237   0.25425079]
 [-1.04235941 -0.6962335  -0.57444585]
 [ 0.8791034  -1.38734296 -0.70762924]]

SKLEARN LINEAR REGRESSION

Intercept:
14.100000000000001

Coefficients:
TV = 3.7641961751469974
Radio = 2.7923066501984364
Newspaper = 0.05597551446265314

Training MSE:
2.705129423081414

Testing MSE:
3.174097353976106

Training R²:
0.8957008271017817

Testing R²:
0.8994380241009119

COST CALCULATION

Initial Parameters:
[0. 0. 0. 0.]

Initial Cost:
112.373125

SCIPY COST MINIMIZATION

Optimized Parameters:
[14.10000001  3.76419614  2.79230693  0.05597505]

Minimum Co